# CoachMe — AI Sports Coach
> "Elite athletes have coaches. Everyone else has CoachMe."

This notebook walks through the full CoachMe pipeline:
1. Connect to Twelve Labs API
2. Create an index with Marengo + Pegasus
3. Upload reference technique videos
4. Upload an athlete video
5. Run similarity search (Marengo)
6. Generate coaching feedback (Pegasus)
7. View everything in FiftyOne

In [ ]:
import os
import time
import fiftyone as fo
from fiftyone import ViewField as F
from twelvelabs import TwelveLabs
from dotenv import load_dotenv

load_dotenv()
print("FiftyOne version:", fo.__version__)

## 1. Connect to Twelve Labs

In [ ]:
client = TwelveLabs(api_key=os.getenv("TWELVELABS_API_KEY"))
print("Connected to Twelve Labs!")

# List existing indexes
for idx in client.index.list():
    print(f"  Index: {idx.name} ({idx.id})")

## 2. Create (or reuse) an index with Marengo + Pegasus

In [ ]:
INDEX_NAME = "coachme-index"

# Check if index already exists
index_id = None
for idx in client.index.list():
    if idx.name == INDEX_NAME:
        index_id = idx.id
        print(f"Reusing existing index: {index_id}")
        break

if not index_id:
    idx = client.index.create(
        name=INDEX_NAME,
        models=[
            {"name": "marengo2.7", "options": ["visual", "audio"]},
            {"name": "pegasus1.2", "options": ["visual", "audio"]},
        ],
    )
    index_id = idx.id
    print(f"Created new index: {index_id}")

print(f"Index ID: {index_id}")

## 3. Upload reference videos

Point `REFERENCE_DIR` at a folder of good-technique videos for your sport.

In [ ]:
# ---- CONFIG: Change these for your demo ----
SPORT = "boxing"  # or squat, tennis, deadlift, etc.
REFERENCE_DIR = "/path/to/reference/videos"  # <-- UPDATE THIS
# ---------------------------------------------

VIDEO_EXT = (".mp4", ".mov", ".avi", ".mkv", ".webm")
ref_videos = sorted(
    os.path.join(REFERENCE_DIR, f)
    for f in os.listdir(REFERENCE_DIR)
    if f.lower().endswith(VIDEO_EXT) and not f.startswith(".")
)
print(f"Found {len(ref_videos)} reference videos")
for v in ref_videos:
    print(f"  {os.path.basename(v)}")

In [ ]:
def upload_and_wait(client, index_id, video_path, timeout_s=600):
    """Upload a video and poll until ready."""
    task = client.task.create(index_id=index_id, file=video_path)
    print(f"  Task {task.id} created for {os.path.basename(video_path)}")
    start = time.time()
    while True:
        t = client.task.retrieve(task.id)
        if t.status == "ready":
            print(f"  Ready! video_id = {t.video_id}")
            return t.video_id
        if t.status == "failed":
            raise RuntimeError(f"Indexing failed for {video_path}")
        if time.time() - start > timeout_s:
            raise TimeoutError("Timed out")
        print(f"  Status: {t.status} ({int(time.time()-start)}s)")
        time.sleep(5)


# Upload all reference videos
ref_video_ids = {}
for vpath in ref_videos:
    print(f"\nUploading {os.path.basename(vpath)}...")
    vid = upload_and_wait(client, index_id, vpath)
    ref_video_ids[vid] = vpath

print(f"\nAll {len(ref_video_ids)} reference videos indexed!")
ref_video_ids

## 4. Save references to FiftyOne dataset

In [ ]:
ds_name = f"coachme-reference-{SPORT}"
if fo.dataset_exists(ds_name):
    ref_dataset = fo.load_dataset(ds_name)
    print(f"Loaded existing dataset: {ds_name} ({len(ref_dataset)} samples)")
else:
    ref_dataset = fo.Dataset(ds_name, persistent=True)
    ref_dataset.tags = ["coachme", "reference", SPORT]
    print(f"Created dataset: {ds_name}")

for vid, fpath in ref_video_ids.items():
    sample = fo.Sample(filepath=fpath)
    sample["sport"] = SPORT
    sample["role"] = "reference"
    sample["tl_video_id"] = vid
    sample["tl_index_id"] = index_id
    ref_dataset.add_sample(sample)

ref_dataset.save()
print(f"Dataset now has {len(ref_dataset)} samples")
ref_dataset

## 5. Upload athlete video

In [ ]:
# ---- CONFIG: Your video ----
ATHLETE_VIDEO = "/path/to/your/video.mp4"  # <-- UPDATE THIS
FOCUS = "overall technique"  # e.g. "elbow position", "footwork"
# ----------------------------

print(f"Uploading athlete video: {os.path.basename(ATHLETE_VIDEO)}")
athlete_video_id = upload_and_wait(client, index_id, ATHLETE_VIDEO)
print(f"Athlete video ID: {athlete_video_id}")

## 6. Marengo Similarity Search

Compare the athlete's video against all reference videos using Marengo embeddings.

In [ ]:
results = client.search.query(
    index_id=index_id,
    query_media_type="video",
    query_media_id=athlete_video_id,
    options=["visual"],
    top_k=10,
)

print("Similarity Results:")
print("=" * 50)

similarity_scores = []
top_matches = []
search_rows = getattr(results, "data", None) or []

for r in search_rows:
    vid_id = getattr(r, "video_id", None) or getattr(r, "id", None)
    score = getattr(r, "score", 0)
    pct = round(score * 100, 1)
    
    is_ref = vid_id in ref_video_ids
    tag = "REF" if is_ref else "SELF"
    name = os.path.basename(ref_video_ids.get(vid_id, "unknown"))
    print(f"  [{tag}] {name}: {pct}%")
    
    if is_ref:
        similarity_scores.append(pct)
        top_matches.append({
            "reference_video_id": vid_id,
            "reference_filepath": ref_video_ids[vid_id],
            "similarity_pct": pct,
        })

avg_score = round(sum(similarity_scores) / len(similarity_scores), 1) if similarity_scores else 0.0
print(f"\nAverage Similarity Score: {avg_score}%")

## 7. Pegasus Coaching Feedback

Ask Pegasus to generate timestamped coaching notes.

In [ ]:
prompt = f"""You are an elite {SPORT} coach reviewing an athlete's technique video.
Focus area: {FOCUS}.

Provide your analysis in this exact format:

## Technique Score: [give a score out of 10]

## What You're Doing Well
- [timestamp] observation

## What Needs Work
- [timestamp] specific issue and how to fix it

## Drill Prescription
List 3 specific drills the athlete should do to improve.

Be direct, specific, and encouraging. Use timestamps like [0:04] throughout."""

print("Generating coaching feedback...")
pegasus_result = client.generate.text(
    video_id=athlete_video_id,
    prompt=prompt,
)

coaching_text = getattr(pegasus_result, "data", None) or str(pegasus_result)
if not isinstance(coaching_text, str):
    coaching_text = str(coaching_text)

print("\n" + "=" * 50)
print("AI COACHING REPORT")
print("=" * 50)
print(coaching_text)

## 8. Save to FiftyOne and browse

In [ ]:
if fo.dataset_exists("coachme-athletes"):
    athlete_ds = fo.load_dataset("coachme-athletes")
else:
    athlete_ds = fo.Dataset("coachme-athletes", persistent=True)
    athlete_ds.tags = ["coachme", "athletes"]

sample = fo.Sample(filepath=ATHLETE_VIDEO)
sample["sport"] = SPORT
sample["role"] = "athlete"
sample["focus_area"] = FOCUS
sample["tl_video_id"] = athlete_video_id
sample["tl_index_id"] = index_id
sample["similarity_score"] = avg_score
sample["reference_matches"] = top_matches
sample["coaching_feedback"] = coaching_text
sample["analyzed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
athlete_ds.add_sample(sample)
athlete_ds.save()

print(f"Saved! Dataset has {len(athlete_ds)} analyzed videos.")
print(f"\nSimilarity Score: {avg_score}%")
print(f"Coaching feedback saved to sample fields.")

In [ ]:
# Launch FiftyOne App to browse results
session = fo.launch_app(athlete_ds)
print("FiftyOne App launched! Open the link above to browse your coaching results.")

## 9. Quick summary

Print a clean summary for the demo.

In [ ]:
print("\n" + "=" * 60)
print("         COACHME — ANALYSIS COMPLETE")
print("=" * 60)
print(f"  Sport:           {SPORT}")
print(f"  Focus:           {FOCUS}")
print(f"  References:      {len(ref_video_ids)} videos")
print(f"  Similarity:      {avg_score}%")
print(f"  Feedback length: {len(coaching_text)} chars")
print(f"  Video ID:        {athlete_video_id}")
print("=" * 60)
print("\n\"Elite athletes have coaches. Everyone else has CoachMe.\"")